<a href="https://colab.research.google.com/github/CristianoMSilva/DistilBERT-FL-Playground/blob/main/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Load the phishing email dataset from Hugging Face Hub
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("zefang-liu/phishing-email-dataset")
print(dataset)

# Convert to pandas for easier inspection
df = dataset["train"].to_pandas()
df.head()

README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv: reconstructing file:   0%|          |  0.00B / 52.0MB            

Phishing_Email.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Email Text', 'Email Type'],
        num_rows: 18650
    })
})


,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [2]:
# Drop the leftover index column from the CSV
df = df.drop(columns=["Unnamed: 0"])

# Class distribution (absolute and percentage)
print(df["Email Type"].value_counts())
print(df["Email Type"].value_counts(normalize=True).round(3))

# Missing values per column
print("\nMissing values:")
print(df.isnull().sum())

# Duplicated emails (same text)
print("\nDuplicated emails:", df.duplicated(subset="Email Text").sum())

Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64
Email Type
Safe Email        0.607
Phishing Email    0.393
Name: proportion, dtype: float64

Missing values:
Email Text    16
Email Type     0
dtype: int64

Duplicated emails: 1112


In [3]:
# Remove empty emails
df = df.dropna(subset=["Email Text"])

# Check duplicates with conflicting labels (same text, different class)
dup_texts = df[df.duplicated(subset="Email Text", keep=False)]
conflicts = dup_texts.groupby("Email Text")["Email Type"].nunique()
print("Texts with conflicting labels:", (conflicts > 1).sum())

# Remove duplicates, keeping the first occurrence
df = df.drop_duplicates(subset="Email Text").reset_index(drop=True)

print("Rows after cleaning:", len(df))
print(df["Email Type"].value_counts())
print(df["Email Type"].value_counts(normalize=True).round(3))

Texts with conflicting labels: 1
Rows after cleaning: 17537
Email Type
Safe Email        10979
Phishing Email     6558
Name: count, dtype: int64
Email Type
Safe Email        0.626
Phishing Email    0.374
Name: proportion, dtype: float64


In [4]:
# Remove the text with conflicting labels (ambiguous ground truth)
conflict_texts = conflicts[conflicts > 1].index
df = df[~df["Email Text"].isin(conflict_texts)].reset_index(drop=True)
print("Rows after removing conflicts:", len(df))

# Email length in words, per class
df["word_count"] = df["Email Text"].str.split().str.len()
print(df.groupby("Email Type")["word_count"].describe().round(1))

# Share of emails likely exceeding DistilBERT's 512-token limit (~380 words)
print("\nEmails > 380 words:", (df["word_count"] > 380).mean().round(3))

Rows after removing conflicts: 17536
                  count   mean      std  min   25%    50%    75%        max
Email Type                                                                 
Phishing Email   6557.0  318.4    587.1  0.0  76.0  145.0  309.0    11625.0
Safe Email      10979.0  698.5  33673.2  1.0  83.0  184.0  396.0  3527576.0

Emails > 380 words: 0.242


In [5]:
# Remove emails with no words (whitespace-only text)
df = df[df["word_count"] > 0].reset_index(drop=True)
print("Rows after removing empty texts:", len(df))

# Inspect the longest emails (possible corrupted rows)
print(df.nlargest(5, "word_count")[["Email Type", "word_count"]])

# Preview the start of the longest one
print("\n", df.loc[df["word_count"].idxmax(), "Email Text"][:500])

Rows after removing empty texts: 17535
       Email Type  word_count
11838  Safe Email     3527576
15102  Safe Email       23343
5245   Safe Email       20004
7068   Safe Email       18971
3610   Safe Email       17960

 0,"Subject: great part-time or summer job ! * * * * * * * * * * * * * * * we have display boxes with credit applications that we need to place in the small owner-operated stores in your area . here is what you do : 1 . introduce yourself to the store owner or manager . 2 . use our 90 % effective script which tells them how this little display box will save their customers hundreds of dollars , be a drawing card for their business , and make them from $ 5 . 00 to $ 15 . 00 or more for every app s


In [6]:
# Remove the corrupted row (embedded CSV content, ~3.5M words)
df = df[df["word_count"] < 1_000_000].reset_index(drop=True)

# Final EDA summary
print("Final rows:", len(df))
print(df["Email Type"].value_counts())
print(df["Email Type"].value_counts(normalize=True).round(3))

print("\nMedian words per class:")
print(df.groupby("Email Type")["word_count"].median())

print("\nEmails > 380 words:", (df["word_count"] > 380).mean().round(3))

Final rows: 17534
Email Type
Safe Email        10978
Phishing Email     6556
Name: count, dtype: int64
Email Type
Safe Email        0.626
Phishing Email    0.374
Name: proportion, dtype: float64

Median words per class:
Email Type
Phishing Email    145.0
Safe Email        183.5
Name: word_count, dtype: float64

Emails > 380 words: 0.242
